# STAR AI — Qwen3.5 on Google Colab GPU

تشغيل نموذج **Qwen3.5** مفتوح الأوزان على GPU سحابي مخصص للجلسة، دون مفتاح Kimi API، ودون استخدام RAM اللابتوب للحوسبة.

**قبل البدء:** من القائمة Runtime → Change runtime type → اختر **GPU** (حسب المتوفر في حسابك) → Save. ثم Run all.

هذا دفتر تفاعلي لا يفتح منفذًا للإنترنت ولا ينشر وكيلًا بعيدًا أو خدمة API عامة. يظل Ollama مربوطًا بـ `127.0.0.1` داخل جهاز Colab فقط. لا تشارك مخرجات حساسة أو مفاتيح هنا.

> Colab Pro ليس خادمًا دائمًا: تخضع جلساته للانتهاء، ولا يضمن نوع GPU بعينه. ولم نفعّل اشتراكًا أو نستهلك موارد حسابك من خلال إنشاء هذا الدفتر.


In [ ]:
import shutil, subprocess, os, json, time
from pathlib import Path
import requests

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
if result.returncode != 0 or not result.stdout.strip():
    raise RuntimeError("No NVIDIA GPU found. In Colab: Runtime → Change runtime type → GPU.")
print("GPU (name / total MiB / free MiB):")
print(result.stdout.strip())

gpu_line = result.stdout.strip().splitlines()[0]
gpu_name, total_mib, free_mib = [p.strip() for p in gpu_line.rsplit(",", 2)]
free_mib = int(free_mib)

# Quantized Ollama models: sizes below exclude context/KV-cache overhead.
if free_mib >= 14000:
    MODEL = "qwen3.5:9b"
elif free_mib >= 8000:
    MODEL = "qwen3.5:4b"
else:
    MODEL = "qwen3.5:2b"
print(f"Selected model: {MODEL}. You can override MODEL before downloading.")
print("Free disk space (GB):", round(shutil.disk_usage("/content").free / 1e9, 1))


In [ ]:
if not shutil.which("ollama"):
    print("Installing Ollama from its official install script...")
    subprocess.run(
        ["bash", "-o", "pipefail", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True
    )
assert shutil.which("ollama"), "Ollama installation did not complete."
print(subprocess.check_output(["ollama", "--version"], text=True).strip())


In [ ]:
OLLAMA_HOST = "127.0.0.1:11434"
local_api = "http://" + OLLAMA_HOST
ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_CONTEXT_LENGTH": "4096",
    "OLLAMA_KEEP_ALIVE": "10m",
})
def ollama_ready():
    try:
        return requests.get(local_api + "/api/tags", timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_ready():
    log_handle = open("/tmp/star-ai-ollama.log", "a")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        env=ollama_env,
        stdout=log_handle, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(40):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            break
        time.sleep(0.5)
if not ollama_ready():
    raise RuntimeError("Ollama did not start. Inspect /tmp/star-ai-ollama.log.")
print("Local Ollama is running on loopback only:", local_api)


In [ ]:
print("Downloading", MODEL, "(first run may download several GB)...")
subprocess.run(["ollama", "pull", MODEL], env=ollama_env, check=True)
print("Model downloaded:", MODEL)


In [ ]:
test = requests.post(
    local_api + "/api/chat",
    json={
        "model": MODEL,
        "stream": False,
        "think": False,
        "messages": [
            {"role": "system", "content": "You are STAR AI. Respond in Arabic and be precise; do not claim to execute commands."},
            {"role": "user", "content": "عرّف نفسك في سطر واحد واذكر اسم نموذجك إن كنت تعرفه."},
        ],
        "options": {"num_ctx": 4096, "num_predict": 256, "temperature": 0.4},
        "keep_alive": "10m",
    },
    timeout=180,
)
test.raise_for_status()
reply = test.json().get("message", {}).get("content", "")
if not reply:
    raise RuntimeError("Model replied with no visible text. Check /tmp/star-ai-ollama.log.")
print("✅ Real model response:\n" + reply)
print("\nGPU/CPU processing status:")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True, env=ollama_env).stdout)


In [ ]:
if "history" not in globals():
    history = [
        {"role": "system", "content": (
            "You are STAR AI, a coding and defensive cyber lab assistant. "
            "Reply in Iraqi Arabic unless asked otherwise. Provide clear code and analysis. "
            "You cannot run terminal commands or claim to inspect systems. "
            "Suggest commands for authorized environments only; changes require owner approval."
        )}
    ]

question = "اكتب لي هيكل مشروع Python احترافي لتحليل سجلات Linux وتنبيه المحاولات المشبوهة."
print("USER:", question)
payload = {"model": MODEL, "messages": history + [{"role": "user", "content": question}],
           "stream": False, "think": False,
           "options": {"num_ctx": 4096, "num_predict": 1000, "temperature": 0.4},
           "keep_alive": "10m"}
res = requests.post(local_api + "/api/chat", json=payload, timeout=300)
res.raise_for_status()
answer = res.json().get("message", {}).get("content", "")
if not answer:
    raise RuntimeError("No answer returned.")
history.extend([{"role": "user", "content": question}, {"role": "assistant", "content": answer}])
# Keep short history to avoid exhausting the GPU's context memory.
history = [history[0]] + history[-8:]
print("\nSTAR AI:\n", answer)


## بعد تشغيل الدفتر

- عدّل متغير `question` في آخر خلية، ثم شغّل الخلية مرة أخرى للمحادثة.
- إذا نقصت الذاكرة، غيّر `MODEL` إلى `qwen3.5:4b` أو `qwen3.5:2b`، ثم أعد تشغيل خليتي التنزيل والاختبار.
- لتخفيف استهلاك GPU بعد الانتهاء شغّل `!ollama stop qwen3.5:9b` فقط إذا كان هذا النموذج هو المحمّل؛ أو أوقف Runtime من Colab.
- اختبار GPU الحقيقي يظهر في مخرجات `ollama ps`؛ تنزيل النموذج أو نجاح المحادثة وحدهما لا يثبتان استخدام GPU.
- **لا تربط عنوان 127.0.0.1 الخاص بـColab بـRailway**: لا يمكن الوصول إليه من الخارج، وجلسة Colab غير دائمة. دمج STAR AI الدائم يحتاج GPU VM مخصصة ومصادقة وربطًا آمنًا.
